In [ ]:
from huggingface_hub import login
from datasets import load_dataset
from datasets import Dataset
from huggingface_hub import login, HfApi
import pandas as pd


import os
os.environ["HF_TOKEN"] = "hf_token" # removed for safety reasons

from huggingface_hub import login
login(os.environ["HF_TOKEN"])

classified_dataset = "businessrules/classified_rules"

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
from datasets import load_dataset

dataset = load_dataset(classified_dataset)
df_syn = dataset["train"].to_pandas()

README.md:   0%|          | 0.00/335 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.89k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/968 [00:00<?, ? examples/s]

In [ ]:
def get_ids_by_has_rule(x):
    matching_ids = []
    for entry in dataset["train"]:
        if entry['label'] == x:
            matching_ids.append(entry['id'])
    return matching_ids
has_rule_ids = get_ids_by_has_rule(0)

In [ ]:
synthetic_dataset = "businessrules/final_dataset_review"
from datasets import load_dataset

dataset_synth = load_dataset(synthetic_dataset)
syn = dataset_synth["train"].to_pandas()

# HDBSCAN + UMAP less number of clusters

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics import silhouette_score
import hdbscan
import umap.umap_ as umap  # Import UMAP

def cluster_hdbscan_codebert_umap_min10(
    df: pd.DataFrame,
    rule_ids: list,
    code_col="cd",
    id_col="id",
    # Tunable UMAP parameters
    n_components=10,   # Target dimensions 
    n_neighbors=15,    # Size of local neighborhood (lower = more local focus)
    min_dist=0.0       # Minimum distance between points in low-dim space
):
    # 1. Filter data
    filtered_df = df[df[id_col].isin(rule_ids)].reset_index(drop=True)
    print(f" Filtered samples: {len(filtered_df)}")

    if filtered_df.empty:
        print("Warning: No data found after filtering.")
        return pd.DataFrame(), None, None, None

    # 2. Load CodeBERT
    model_name = "microsoft/codebert-base"
    print(f"Loading embedding model: {model_name}")
    model = SentenceTransformer(model_name)

    # 3. Embed code using CodeBERT (Result: 768 dimensions)
    print("Generating raw embeddings (768-d)...")
    raw_embeddings = model.encode(
        filtered_df[code_col].tolist(),
        batch_size=32,
        show_progress_bar=True
    )

    # 4. Dimensionality Reduction with UMAP (CRITICAL STEP)
    #    This projects the 768-dim fog into a clear lower-dimensional map
    print(f"Applying UMAP to reduce dimensions to {n_components}...")
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        n_components=n_components,
        min_dist=min_dist,
        metric='cosine',  
        random_state=42   # Fixed seed for reproducibility
    )
    umap_embeddings = reducer.fit_transform(raw_embeddings)

    # 5. Run HDBSCAN on the UMAP embeddings
    print(" Running HDBSCAN on UMAP embeddings...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=10,  # Smallest size to consider a cluster
        min_samples=3,       
        metric='euclidean',  
        cluster_selection_method='eom'
    )

    clusters = clusterer.fit_predict(umap_embeddings)

    # Add cluster labels to dataframe
    filtered_df["cluster_id"] = clusters

    # 6. Compute silhouette score
    # We calculate score on the UMAP embeddings to see how well separated the manifolds are.
    mask = clusters != -1
    if mask.sum() > 1:
        score = silhouette_score(umap_embeddings[mask], clusters[mask], metric='euclidean')
    else:
        score = -1

    print(f"\n Silhouette Score (on UMAP features): {score:.4f}")

    print("\n Cluster distribution:")
    print(filtered_df["cluster_id"].value_counts().sort_index())

    return filtered_df, umap_embeddings, clusters, score

In [ ]:
clustered_umap2, embeddings_umap2, cluster_ids_umap2, score_umap2 = cluster_hdbscan_codebert_umap_min10(
    df=syn,
    rule_ids=has_rule_ids,
    code_col="cd",
    id_col="id",
)

In [ ]:
print(f"\n Silhouette Score (on UMAP features): {score_umap2:.4f}")

print("\n Cluster distribution:")
print(clustered_umap2["cluster_id"].value_counts().sort_index())

In [ ]:
import pandas as pd
import numpy as np

def get_stratified_id_lists(df: pd.DataFrame, id_col: str, cluster_col: str = "cluster_id"):
    """
    Groups the DataFrame by cluster ID and extracts a list of code IDs for each cluster.
    """
    id_lists = {}

    # Iterate through each unique cluster ID
    for cluster_id, group in df.groupby(cluster_col):
        # Convert the column of IDs for that group to a standard Python list
        id_lists[cluster_id] = group[id_col].tolist()

    # Separate the noise cluster (-1) for explicit handling
    noise_ids = id_lists.pop(-1, [])

    print(f" Extracted IDs for {len(id_lists)} main clusters.")
    print(f" Extracted {len(noise_ids)} IDs from the Noise cluster (-1).")

    return id_lists, noise_ids

In [ ]:
stratified_ids, noise_ids = get_stratified_id_lists(
    df=clustered_umap2,   
    id_col="id",          
    cluster_col="cluster_id" 
)

print(f"Total functional clusters found: {len(stratified_ids)}")
print(f"Example: First 5 IDs in Cluster 0: {stratified_ids.get(0, [])[:5]}")

# No Rule Data points

In [ ]:
def get_ids_by_no_rule(x):
    matching_ids = []
    for entry in dataset["train"]:
        if entry['label'] == x:
            matching_ids.append(entry['id'])
    return matching_ids
no_rule_ids = get_ids_by_no_rule(1)

# Code Type 1 & 2

In [ ]:
def get_ids_by_code_lang(x):
    """
    Filters the dataset to find all 'id's where 'code lang' matches the given value x.

    Args:
        x: The value to match in the 'code lang' column.

    Returns:
        A list of 'id' values that match the criteria.
    """
    matching_ids = []
    for entry in dataset_synth["train"]:
        if str(entry['code lang']) == str(x):
            matching_ids.append(entry['id'])
    return matching_ids

In [ ]:
code_type_1_ids = get_ids_by_code_lang(1.0)
print(code_type_1_ids)

[26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 40, 375]


In [ ]:
code_type_2_ids = get_ids_by_code_lang(2.0)
print(code_type_2_ids)

[43, 44, 45, 46, 47, 48, 49, 50, 65, 113, 235, 549, 716, 731, 904, 913, 936]


# Stratified Sampling

In [ ]:
from sklearn.model_selection import train_test_split
import random
import math

def perform_stratified_and_independent_splits_unified(
    clustered_df: pd.DataFrame,
    other_id_lists: list,
    test_size: float = 0.20,
    random_seed: int = 42
):
    """
    1. Splits the main clustered/noise IDs using a single stratified call.
    2. Splits the independent lists using simple random sampling,
       with the training size rounded up (ceil).
    3. Consolidates all train IDs and all test IDs.
    """
    random.seed(random_seed) 
    final_train_ids = []
    final_test_ids = []

    id_col = 'id'
    cluster_col = 'cluster_id'

    ### A. Unified Stratified Split (Main Dataset + Noise) ###


    # 1. NO SEPARATION: Use the entire DataFrame (clustered data + noise data)
    main_data = clustered_df.copy()

    # 2. Perform Stratified Split on the ENTIRE dataset
    train_main, test_main = train_test_split(
        main_data,
        test_size=test_size,
        stratify=main_data[cluster_col],
        random_state=random_seed
    )

    # Consolidate train/test lists from the main clustered data
    final_train_ids.extend(train_main[id_col].tolist())
    final_test_ids.extend(test_main[id_col].tolist())

    print(f" Stratified/Unified Split Complete: {len(final_train_ids)} train, {len(final_test_ids)} test (Initial)")

    ### B. Simple Random Split (for the 3 other ID lists) ###

    for i, id_list in enumerate(other_id_lists):
        if not id_list:
            print(f"List {i+1} is empty, skipping.")
            continue

        # --- NEW LOGIC START ---
        total_size = len(id_list)
        # Calculate the required training size (1 - test_size)
        train_proportion = 1.0 - test_size

        # Calculate the number of items needed for training, rounded UP (ceil)
        # Example: 17 * 0.8 = 13.6 -> ceil(13.6) = 14
        train_size_rounded_up = math.ceil(total_size * train_proportion)

        # Calculate the corresponding test size
        test_size_calculated = total_size - train_size_rounded_up

        # Use train_test_split with the calculated 'train_size'
        train_list, test_list = train_test_split(
            id_list,
            train_size=train_size_rounded_up, 
            test_size=test_size_calculated,   
            random_state=random_seed,
        )
        # --- NEW LOGIC END ---

        final_train_ids.extend(train_list)
        final_test_ids.extend(test_list)
        print(f"  -> List {i+1} added: {len(train_list)} train ({train_size_rounded_up} requested), {len(test_list)} test ({test_size_calculated} requested).")

    ### C. Final Consolidation and Verification ###


    # Remove duplicates if any IDs overlap across lists
    final_train_ids = sorted(list(set(final_train_ids)))
    final_test_ids = sorted(list(set(final_test_ids)))

    # Quick check for overlaps
    overlap = set(final_train_ids) & set(final_test_ids)
    if overlap:
        print(f"❌ WARNING: {len(overlap)} IDs are in both train and test sets! Review source data.")

    print(f"\n✨ FINAL TOTAL SETS ✨")
    print(f"Total Train IDs: {len(final_train_ids)}")
    print(f"Total Test IDs: {len(final_test_ids)}")
    print(f"Total IDs: {len(final_train_ids) + len(final_test_ids)}")

    return final_train_ids, final_test_ids

In [ ]:
other_lists = [no_rule_ids, code_type_1_ids, code_type_2_ids]

final_train, final_test = perform_stratified_and_independent_splits_unified(
    clustered_umap2,
    other_lists
)

✅ Stratified/Unified Split Complete: 729 train, 183 test (Initial)
  -> List 1 added: 45 train (45 requested), 11 test (11 requested).
  -> List 2 added: 12 train (12 requested), 3 test (3 requested).
  -> List 3 added: 14 train (14 requested), 3 test (3 requested).

✨ FINAL TOTAL SETS ✨
Total Train IDs: 800
Total Test IDs: 200
Total IDs: 1000


In [ ]:
print(len(no_rule_ids))
print(len(code_type_1_ids))
print(len(code_type_2_ids))
print(len(has_rule_ids))


56
15
17
912


# Split on the Hugging Face Dataset

In [ ]:
from datasets import load_dataset

repo_name = "businessrules/final_dataset_review"
ds = load_dataset(repo_name)


In [ ]:
base_ds = ds["train"]

In [ ]:
def split_by_id(dataset, train_ids, test_ids, id_col="id"):
    train_set = dataset.filter(lambda x: x[id_col] in train_ids)
    test_set  = dataset.filter(lambda x: x[id_col] in test_ids)
    return train_set, test_set

train_ds, test_ds = split_by_id(base_ds, final_train, final_test)

In [ ]:
from datasets import DatasetDict

split_ds = DatasetDict({
    "train": train_ds,
    "test": test_ds
})

In [ ]:
split_repo = "businessrules/dataset_stratified_test"

split_ds.push_to_hub(split_repo)


# Data Fixture

In [ ]:
combined_ids = sorted(list(set(final_train + final_test)))

print(f"Total unique combined IDs: {len(combined_ids)}")
print(f"First 10 combined IDs: {combined_ids[:10]}")

# Find missing IDs between 1 and 100
all_possible_ids = set(range(1, 1001))
present_ids_in_range = set(id for id in combined_ids if 1 <= id <= 1000)

missing_ids = sorted(list(all_possible_ids - present_ids_in_range))

print(f"\nIDs missing between 1 and 100: {missing_ids}")
print(f"Number of missing IDs: {len(missing_ids)}")

Total unique combined IDs: 998
First 10 combined IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

IDs missing between 1 and 100: [367, 373]
Number of missing IDs: 2


In [ ]:
from datasets import load_dataset

# ---------------------------------------------------------
# 1. Load dataset
# ---------------------------------------------------------
data_fix1 = load_dataset("businessrules/final_dataset_review", split="train")

# IDs you want to fix
TARGET_IDS = {367, 373}

# ---------------------------------------------------------
# 2. Conditional update (row-wise, safe)
# ---------------------------------------------------------
def fix_code_lang(example):
    if example["id"] in TARGET_IDS:
        example["code lang"] = 0
    return example

dataset_fixed = data_fix1.map(fix_code_lang)

# ---------------------------------------------------------
# 3. Push back to HF (overwrites repo, but NOT rows)
# ---------------------------------------------------------
dataset_fixed.push_to_hub("businessrules/final_dataset_review")


In [ ]:
from datasets import load_dataset, Dataset, concatenate_datasets

# ---------------------------------------------------------
# 1. Load dataset
# ---------------------------------------------------------
fix_data2 = load_dataset("businessrules/classified_rules", split="train")

# ---------------------------------------------------------
# 2. Define new rows
# ---------------------------------------------------------
new_rows = [
    {"id": 367, "label": 0, "label_name": "HAS_RULE"},
    {"id": 373, "label": 0, "label_name": "HAS_RULE"},
]

new_ids = {row["id"] for row in new_rows}

# ---------------------------------------------------------
# 3. Safety check (NO overrides)
# ---------------------------------------------------------
existing_ids = set(fix_data2["id"])

conflicts = new_ids.intersection(existing_ids)
if conflicts:
    raise ValueError(f"IDs already exist in dataset: {conflicts}")

# ---------------------------------------------------------
# 4. Create Dataset for new rows (schema-aligned)
# ---------------------------------------------------------
new_dataset = Dataset.from_list(new_rows, features=fix_data2.features)

# ---------------------------------------------------------
# 5. Append rows (non-destructive)
# ---------------------------------------------------------
updated_dataset = concatenate_datasets([fix_data2, new_dataset])

# ---------------------------------------------------------
# 6. Push back to HF
# ---------------------------------------------------------
updated_dataset.push_to_hub("businessrules/classified_rules")
